# データベース演習 第10回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- **「編集者」ではなく「閲覧者」**に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

### 使用するデータベースのダウンロード

* 今回は，サイズの大きなweblogデータベースを使用します（ファイル名：weblog_l.sqlite3）．
* access_logの全データがありますが，access_log_wideは含まれません．
* これまでは，githubからダウンロードしていましたが，サイズの関係でGoogle Driveからダウンロードします．
* Colabのランタイムが切り替わると，その度にダウンロードする必要があります
* その度にダウンロードするのが面倒な人は，自分のGoogleドライブに保存し，そのファイルを参照する方法もあります（生成AIなどで調べてみてください）．

### データベースをダウンロード

In [ ]:
# Google DriveからSQLiteファイルを取得
import gdown

# ファイルIDを指定
file_id = '1QGxNjQNHvjh0mgFY--HkVTeSEC_bTMHX'
url = f'https://drive.google.com/uc?id={file_id}'

# 保存ファイル名を指定
output = 'weblog_l.sqlite3'
gdown.download(url, output, quiet=False)

### JupySQLのインストールと有効化

In [ ]:
# Colabではjupysqlのインストールが必要
import sys
if 'google.colab' in sys.modules:
    %pip install -q jupysql

In [ ]:
# jupysqlの拡張機能を有効化`
%load_ext sql

### Webログデータベース

In [ ]:
# Webログデータベースに接続する
%sql sqlite:///weblog_l.sqlite3

### 含まれるテーブルの確認

In [ ]:
%%sql
SELECT * FROM sqlite_master;


## 例題1

customersテーブルに対して，以下のようなSQL文を作成し実行せよ
（行数が多いため，すべて結果の上限を10行に制限すること）

(1)（すべての）customer_ageの値に100を加えた結果を求めるSQL文

In [ ]:
%%sql
SELECT customer_age + 100 FROM customers LIMIT 10;

(2) customer_ageに対して，演算子%を使用して年齢層を求めるSQL文

In [ ]:
%%sql
SELECT customer_age - (customer_age%10) FROM customers LIMIT 10;

(3) customer_birthdayの値の日のみを求めるSQL文（1978-08-22なら22を求める）

In [ ]:
%%sql
SELECT substring(cast(customer_birthday AS text), 9, 2) FROM customers LIMIT 10;

## 例題2

web_pagesテーブルは，ウェブページのアドレス（カラム名request_path，例：/index.html）と人間に理解できるウェブページの名前（カラム名 page_name，例：top）を対応づけるテーブルである．一方access_logテーブルは，ウェブページのアドレスにアクセスした時間（カラム名 request_time）や顧客ID（カラム名 customer_id）を管理する．以下のようなSQL文を作成し実行せよ
（行数が多いため，すべて結果の上限を10行に制限すること）

何時（request_time），誰（customer_id）が，どの名前のページ(page_name)にアクセスしているかを抽出したい．上の2つのテーブルを，ウェブページのアドレス（request_path）が等しいという条件でジョインし，前記の3つのカラムを取得するSQL文を作成せよ（カラムの並びはこの順とする）

In [ ]:
%%sql
SELECT a.request_time, a.customer_id, p.page_name FROM access_log AS a
JOIN web_pages AS p on a.request_path = p.request_path
LIMIT 10;

## 演習 課題5-1

itemsテーブルに対して，以下のようなSQL文を作成せよ
（行数が多いため，すべて結果の上限を10行に制限すること）

(1) （すべての）shop_idの値に100を加えた結果を求めるSQL文


(2) item_priceの値から，1000円未満の値を切り捨てた値を求めるSQL文
（例：item_priceが7800であれば，7000を求める，同様に13010→13000を求める）


(3) item_nameの値の最初の5文字だけを求めるSQL文

## 演習 課題5-2

以下のようなSQL文を作成せよ
（行数が多いため，すべて結果の上限を10行に制限すること）

(4) 例題2において，topページに関する情報のみがほしい（page_nameがtop）．topページに限定して，例題2の3つのカラム（request_time, customer_id, page_name）を取得するSQL文を作成せよ

(5) 例題2において，誰にあたる情報をidではなく名前（customer_name）にしたい．例題2の2テーブルに加え，customersテーブルを利用し，3テーブルのジョインを用いて，request_time, customer_name, page_nameの3つのカラムを取得するSQL文を作成せよ